In [32]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [33]:
dataset = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [34]:
# Lowercasing : 

dataset['review'] = dataset['review'].str.lower()
dataset.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


When regx is used and when BeautifulScoup ?

“I use regex when the HTML is simple, predictable, and performance is critical—like removing basic tags in large datasets using vectorized Pandas operations. However, for complex or nested HTML structures, BeautifulSoup is more reliable because regex cannot correctly parse hierarchical data.”

In [35]:
# Remove 'html' tags : using BeautifulScoup 

from  bs4 import BeautifulSoup

dataset['review'] = dataset['review'].apply(lambda x : BeautifulSoup(x , 'html.parser').get_text())

dataset.head()


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [36]:
# Remove 'html' tags : using Regx(regular expression)

import re

def remove_html(text):
    if not isinstance(text, str):
        return text
    pattern = re.compile(r'<[^>]+>')
    return pattern.sub('', text)

dataset['review'] = dataset['review'].apply(remove_html)

In [37]:
# Remove 'URLs' : “I use the pattern https?://\S+|www\.\S+ to capture both HTTP/HTTPS and www-based URLs, removing everything until the next whitespace.”

def remove_url(text):
    if not isinstance(text, str):
        return text
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub('', text)

dataset['review'] = dataset['review'].apply(remove_url)

In [38]:
# Remove 'Punctuation' :

import string

def remove_punc(text):
    if not isinstance(text , str):
        return text
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

dataset['review'] = dataset['review'].apply(remove_punc)

In [39]:
dataset.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [40]:
# Chat word treatment : when we have this type of data :
chat_words = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ASL": "Age, Sex, Location",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can’t Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don’t Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "PITA": "Pain In The A**",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn’t Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTF": "What The F**k",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired"
}

In [41]:
# Chat word treatment : ex : replace Gn with Good night 

def chat_conversion(text) :
    new_text = []
    for w in text.split():
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)

dataset['review'] = dataset['review'].apply(chat_conversion)

In [42]:
chat_conversion('IMHO is the best')

'In My Honest/Humble Opinion is the best'

In [ ]:
# Spelling Correction :

from textblob import TextBlob

def correct_text(text):
    if not isinstance(text, str):
        return text
    return str(TextBlob(text).correct())

dataset['review'] = dataset['review'].apply(correct_text)

In [ ]:
# example 

incorrect = 'ceertain condition during sevrl gneration are modifred in ths same mannerr.'
textblb = TextBlob(incorrect)

textBlb.correct().string

In [47]:
# Removing Stop words :  ex : the , are , my , a etc

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    
    words = text.split()  # simple tokenization
    filtered = [word for word in words if word.lower() not in stop_words]
    
    return " ".join(filtered)

dataset['review'] = dataset['review'].apply(remove_stopwords)

In [48]:
dataset.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend Tears Eyes hot sum...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love Tears Eyes money visually ...,positive


In [ ]:
# Handling 'Emoji' :

# Remove Emoji :

import re
def remove_emoji(text):
    if not isinstance(text, str):
        return text
    
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002700-\U000027BF"  # dingbats
        u"\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    
    return emoji_pattern.sub('', text)

In [49]:
#  Replace emoji :

import emoji

def replace_emoji(text):
    if not isinstance(text, str):
        return text
    return emoji.demojize(text, delimiters=(" ", " "))

In [51]:
dataset['review'] = dataset['review'].apply(replace_emoji)

In [56]:
# Tokenization1 : word tokenization

sent1 = 'i am going to new delhi'
sent1.split()

['i', 'am', 'going', 'to', 'new', 'delhi']

In [61]:
# Tokenization2 : sentence tokenization 

sent2 = "I am going to delhi.I will stay there for 3 days .Lets\'s hope the trip to be great."
sent2.split(".")

['I am going to delhi',
 'I will stay there for 3 days ',
 "Lets's hope the trip to be great",
 '']

In [60]:
# Problem with split fxn :

sent3 = "I am going to delhi!"
sent3.split()

['I', 'am', 'going', 'to', 'delhi!']

In [68]:
# Tokenization : using spacy lib (best and imp)


import spacy
nlp = spacy.load("en_core_web_sm")

def tokenize_spacy(text):
    if not isinstance(text, str):
        return text
    
    doc = nlp(text)
    return [token.text for token in doc]

In [65]:
# Tokenization : split text into words : IMportant (Recommended) using nltk lib

from nltk.tokenize import word_tokenize , sent_tokenize

def tokenize(text):
    if not isinstance(text, str):
        return text
    return word_tokenize(text)

In [66]:
dataset['review'] = dataset['review'].apply(tokenize)

In [67]:
dataset.head()

,review,sentiment
0,"[one, reviewers, mentioned, watching, 1, oz, e...",positive
1,"[wonderful, little, production, filming, techn...",positive
2,"[thought, wonderful, way, spend, Tears, Eyes, ...",positive
3,"[basically, theres, family, little, boy, jake,...",negative
4,"[petter, matteis, love, Tears, Eyes, money, vi...",positive


In [69]:
# Stemming : reducing word to their root/base form. uses in information retrieval systems such as search engines .

# using nltk PortStemmer 

from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stemming_nltk(text):
    if not isinstance(text, str):
        return text
    
    words = text.split()
    stemmed_words = [ps.stem(word) for word in words]
    
    return " ".join(stemmed_words)



In [70]:
# using stemming  with tokenizer :

from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

ps = PorterStemmer()

def stemming2(text):
    if not isinstance(text, str):
        return text
    
    words = word_tokenize(text)
    return " ".join([ps.stem(w) for w in words])

In [75]:
# dataset['review'] = dataset['review'].apply(stemming_nltk)

In [77]:
# Lemitization : reduces word to thier root insuring that the root word belong to the language

# using nltk :

from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    if not isinstance(text , str):
        return text
    words = word_tokenize(text)

    return " ".join([lemmatizer.lemmatize(w) for w in words])


In [78]:
# using spacy : Imp / Best 

import spacy

nlp = spacy.load('en_core_web_sm')

def lemmatize_text2(text):
    if not isinstance(text, str):
        return text

    doc = nlp(text)
    return " ".join([token.lemma_ for token in dox])

lemmetization our review using spacy :

In [ ]:
dataset['review'] = dataset['review'].apply(
    lambda words: [token.lemma_ for token in nlp(" ".join(words))]
)

In [ ]:
dataset.head()